# Task 2 – Model Training & Evaluation

This notebook trains and evaluates regression models to predict `SpendingScore` from `Gender`, `Age`, and `AnnualIncome`.


In [ ]:
# 1. Imports
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


In [ ]:
# 2. Load cleaned dataset
csv_path = "cleaned_spending_data.csv"
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
df.head()


In [ ]:
# 3. Define features and target
FEATURES = ["Gender", "Age", "AnnualIncome"]
TARGET_COL = "SpendingScore"

X = df[FEATURES].copy()
y = df[TARGET_COL].copy()

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


In [ ]:
# 5. Preprocessor
categorical_features = ["Gender"]
numeric_features = ["Age", "AnnualIncome"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ]
)


In [ ]:
# 6. Define models
linreg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])


In [ ]:
# 7. Helper function to evaluate a model
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)
    print(name)
    print(f"  MAE  : {mae:.3f}")
    print(f"  RMSE : {rmse:.3f}")
    print(f"  R^2  : {r2:.3f}")
    print("-" * 40)
    return {"name": name, "mae": mae, "rmse": rmse, "r2": r2, "model": model}

# 8. Train & evaluate both models
lin_metrics = evaluate_model("Linear Regression", linreg_model, X_train, y_train, X_test, y_test)
rf_metrics = evaluate_model("Random Forest Regressor", rf_model, X_train, y_train, X_test, y_test)


In [ ]:
# 9. Select best model
best = rf_metrics if rf_metrics["r2"] > lin_metrics["r2"] else lin_metrics
best_name = best["name"]
best_model = best["model"]

print("Best model based on R^2:", best_name)

# 10. Save best model to artifacts
artifacts_dir = "artifacts"
os.makedirs(artifacts_dir, exist_ok=True)
model_path = os.path.join(artifacts_dir, "spending_model.pkl")
joblib.dump(best_model, model_path)
print("Saved best model to:", model_path)
